# `rmgdb` and `rmgdatabase`

This demo notebook shows how to use the SQL-wrapped version of RMG-database to access all of the various data contained within.

`standard` holds the `rmgdb` package, which specifies the actual layout of the database.
`data` holds the `rmgdatabase` package, which uses `rmgdb` to build an actual database file from the `RMG-database` Python source files.
`data` also includes a plaintext dump of `rmdatabase` into the YAML format - this is easier to read and more broadly intercompatible with other programming tools than the original Python files in `RMG-database`.
This demo won't cover using these files, but they are available.

There are five sub-databases:

 1. kinetics
 2. solvation
 3. statmech
 4. thermo
 5. transport

Each has libraries (collated data from the literature) and families (RMG-specific subsets from the libraries).

Running this notebook to access the data only requires one dependency: `pandas`.
No database setup is needed; Python includes `sqlite3` in its standard library, which runs without any additional complications.

One could also substitute `pandas` for `polars`, `narwhals`, etc. - really any library that can read from a SQL database.
Enterprising users may also see fit to just use `sqlite3` directly and avoid dependencies altogether, but this requires a more advanced understanding of writing SQL queries.

In [1]:
import pandas as pd

Some quick background to help make this notebook make sense: much of RMG-database stores data as shown below in this random example from the solvation sub-database.

```python
entry(
    index = 2,
    label = "propane",
    molecule = "CCC",
    solute = SoluteData(
        S = 0,
        B = 0,
        E = 0,
        L = 1.05,
        A = 0,
        V = 0.5313,
    ),
    shortDesc = """""",
    longDesc =
"""
From Abarahm et al., J. Chem. Soc., Perkin Trans. 2, 1994, 1777-1791,
DOI: 10.1039/P29940001777
""",
)
```

You can see that the `SoluteData` class is __nested__ inside our call to `entry`.
This is strictly forbidden in SQL databases; instead we store all of the calls to `entry` in one table, all of the calls to `SoluteData` in another, and then provide a 'lookup key' to match the two of them back up.

This process of re-joining the two tables is cumbersome and requires knowing some SQL.
To avoid that, SQL supports __views__ - these are basically just queries against the database that you can treat like regular tables.
They avoid you having to write SQL statements to 'rebuild' the totally flat version of the database.

All of the examples below use the various views to retrieve data.
You can of course directly look at the tables, but there's no need (unless you want to write your own SQL, in which case I suggest familiarizing yourself with the schema in `standard`).
The below function is not needed to actually  use `rmgdatabase`, but is included to help the demo.

In [2]:
import sqlite3

def list_all_views(database_file):
    """
    Connects to an SQLite database and lists all views.
    
    Args:
        database_file (str): The path to the SQLite database file.
    
    Returns:
        list: A list of view names.
    """
    conn = None
    try:
        # Create a database connection
        conn = sqlite3.connect(database_file)
        cursor = conn.cursor()

        # Query the sqlite_master table for views
        cursor.execute("SELECT name FROM sqlite_master WHERE type='view' ORDER BY name;")
        
        # Fetch all results
        views = cursor.fetchall()

        # Print the results
        if views:
            print(f"Views in database '{database_file}':")
            for view in views:
                print(f"- {view[0]}")
        else:
            print(f"No views found in database '{database_file}'.")
        
        # Return the list of view names
        return [view[0] for view in views]

    except sqlite3.Error as e:
        print(f"An error occurred: {e}")
        return []
    finally:
        # Close the connection
        if conn:
            conn.close()


One final note - RMG uses its own format for storing molecular structures called the 'adjacency list'.
These can be converted into more friendly formats (INCHI, SMILES) using RMG-Py.

## `transport`

Let's start by looking at what views we have:

In [3]:
transport_db = "demo_db/transport.db"
list_all_views(transport_db);


Views in database 'demo_db/transport.db':
- label_pairs_view
- transport_groups_view
- transport_libraries_view


The `libraries` view contains data from the literature, digitized into `rmgdatabase`.
The `groups` view contains the actual substructures used by RMG to make estimations for transport properties, with the `label_pairs` view showing how the rows of that table are related to one another in the tree structure.
This demo is focused on just getting data out of `rmgdatabase` - future work can look toward re-building the estimator tree and integrating with RMG-Py.

Let's open up the transport libraries:

In [4]:
pd.read_sql("""SELECT * from transport_libraries_view""", "sqlite:///" + transport_db).set_index("id")

,name,short_description,long_description,label,adjacency_list,shapeIndex,epsilon,epsilon_unit,sigma,sigma_unit,dipoleMoment,dipoleMoment_unit,polarizability,polarizability_unit,rotrelaxcollnum
id,,,,,,,,,,,,,,,
0,OneDMinN2,,,N2,"\n 1 N u0 p1 c0 {2,T}\n 2 N u0 p1 c0 {1,...",1,322.846,K,3.461,angstroms,1.781,De,0.000,angstroms^3,1.0
1,OneDMinN2,,,NNH,"\n multiplicity 2\n 1 N u1 p1 c0 {2,D}\n...",1,292.088,K,3.459,angstroms,1.858,De,2.016,angstroms^3,1.0
2,OneDMinN2,,,H2NN(S),"\n 1 N u0 p0 c+1 {2,S} {3,S} {4,D}\n 2 H...",2,387.557,K,3.467,angstroms,3.507,De,2.349,angstroms^3,1.0
3,OneDMinN2,,,H2NN(T),"\n multiplicity 3\n 1 N u0 p1 c0 {2,S} {...",2,338.122,K,3.528,angstroms,2.363,De,2.371,angstroms^3,1.0
4,OneDMinN2,,,N2H2,"\n 1 N u0 p1 c0 {2,D} {3,S}\n 2 N u0 p1 ...",2,323.008,K,3.531,angstroms,0.000,De,2.297,angstroms^3,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
382,NOx2018,,,H2NCHO,"\n 1 N u0 p1 c0 {2,S} {4,S} {5,S}\n 2 C ...",2,307.800,K,4.140,angstroms,0.000,De,0.000,angstroms^3,1.0
383,NOx2018,,,H2NCO,"\n multiplicity 2\n 1 N u0 p1 c0 {2,S} {...",2,307.800,K,4.140,angstroms,0.000,De,0.000,angstroms^3,1.0
384,NOx2018,,,CH3NC,"\n multiplicity 3\n 1 C u0 p0 c0 {2,S} {...",2,422.220,K,5.329,angstroms,3.500,De,0.000,angstroms^3,1.0


We can now see all of the columns that are available - we probably only want to use a subset of these, so here's a function to do so:

In [5]:
def read_sql(view_name, database_file, columns=None):
    return pd.read_sql(f"""SELECT {', '.join(columns) if columns else '*'} from {view_name}""", "sqlite:///" + database_file)

In [6]:
read_sql("transport_libraries_view", transport_db, columns=["id", "name", "adjacency_list", "epsilon", "sigma"]).set_index("id")

,name,adjacency_list,epsilon,sigma
id,,,,
0,OneDMinN2,"\n 1 N u0 p1 c0 {2,T}\n 2 N u0 p1 c0 {1,...",322.846,3.461
1,OneDMinN2,"\n multiplicity 2\n 1 N u1 p1 c0 {2,D}\n...",292.088,3.459
2,OneDMinN2,"\n 1 N u0 p0 c+1 {2,S} {3,S} {4,D}\n 2 H...",387.557,3.467
3,OneDMinN2,"\n multiplicity 3\n 1 N u0 p1 c0 {2,S} {...",338.122,3.528
4,OneDMinN2,"\n 1 N u0 p1 c0 {2,D} {3,S}\n 2 N u0 p1 ...",323.008,3.531
...,...,...,...,...
382,NOx2018,"\n 1 N u0 p1 c0 {2,S} {4,S} {5,S}\n 2 C ...",307.800,4.140
383,NOx2018,"\n multiplicity 2\n 1 N u0 p1 c0 {2,S} {...",307.800,4.140
384,NOx2018,"\n multiplicity 3\n 1 C u0 p0 c0 {2,S} {...",422.220,5.329


LLMs are generally _very_ good at writing small functions like this, so they are highly recommended for this application.
This demo contains a number of useful functions for loading the sub-databases, as well.

One can also just load the _entire_ view into memory with pandas, and then throw data out as needed, though this may be less efficient.
For example, here's a query that loads only a subset of the columns (renaming one of them, just for fun) with the additional requirement that `epsilon` and `adjacency_list` are present:

In [7]:
pd.read_sql("""
    SELECT name as library_name, adjacency_list, sigma, sigma_unit 
    FROM transport_libraries_view 
    WHERE epsilon IS NOT NULL AND adjacency_list IS NOT NULL
""", "sqlite:///" + transport_db)


,library_name,adjacency_list,sigma,sigma_unit
0,OneDMinN2,"\n 1 N u0 p1 c0 {2,T}\n 2 N u0 p1 c0 {1,...",3.461,angstroms
1,OneDMinN2,"\n multiplicity 2\n 1 N u1 p1 c0 {2,D}\n...",3.459,angstroms
2,OneDMinN2,"\n 1 N u0 p0 c+1 {2,S} {3,S} {4,D}\n 2 H...",3.467,angstroms
3,OneDMinN2,"\n multiplicity 3\n 1 N u0 p1 c0 {2,S} {...",3.528,angstroms
4,OneDMinN2,"\n 1 N u0 p1 c0 {2,D} {3,S}\n 2 N u0 p1 ...",3.531,angstroms
...,...,...,...,...
382,NOx2018,"\n 1 N u0 p1 c0 {2,S} {4,S} {5,S}\n 2 C ...",4.140,angstroms
383,NOx2018,"\n multiplicity 2\n 1 N u0 p1 c0 {2,S} {...",4.140,angstroms
384,NOx2018,"\n multiplicity 3\n 1 C u0 p0 c0 {2,S} {...",5.329,angstroms
385,NOx2018,"\n multiplicity 2\n 1 C u0 p0 c0 {2,S} {...",4.860,angstroms


And here's the same, but in Pandas:

In [8]:
df_transport_all = pd.read_sql("SELECT * FROM transport_libraries_view", "sqlite:///" + transport_db).set_index("id")
df_transport = df_transport_all[
    df_transport_all['epsilon'].notna() & 
    df_transport_all['adjacency_list'].notna()
].copy()
df_transport.rename(columns={'name': 'library_name'}, inplace=True)
df_transport[['library_name', 'adjacency_list', 'sigma', 'sigma_unit']]

,library_name,adjacency_list,sigma,sigma_unit
id,,,,
0,OneDMinN2,"\n 1 N u0 p1 c0 {2,T}\n 2 N u0 p1 c0 {1,...",3.461,angstroms
1,OneDMinN2,"\n multiplicity 2\n 1 N u1 p1 c0 {2,D}\n...",3.459,angstroms
2,OneDMinN2,"\n 1 N u0 p0 c+1 {2,S} {3,S} {4,D}\n 2 H...",3.467,angstroms
3,OneDMinN2,"\n multiplicity 3\n 1 N u0 p1 c0 {2,S} {...",3.528,angstroms
4,OneDMinN2,"\n 1 N u0 p1 c0 {2,D} {3,S}\n 2 N u0 p1 ...",3.531,angstroms
...,...,...,...,...
382,NOx2018,"\n 1 N u0 p1 c0 {2,S} {4,S} {5,S}\n 2 C ...",4.140,angstroms
383,NOx2018,"\n multiplicity 2\n 1 N u0 p1 c0 {2,S} {...",4.140,angstroms
384,NOx2018,"\n multiplicity 3\n 1 C u0 p0 c0 {2,S} {...",5.329,angstroms


## `thermo`

Once more, let's look at the views:

In [9]:
thermo_db = "demo_db/thermo.db"
list_all_views(thermo_db);


Views in database 'demo_db/thermo.db':
- label_pairs_view
- thermo_depositories_view
- thermo_groups_view
- thermo_libraries_view


Much the same story as the `transport` sub-database, with the only addition being the `depositories` view (mean for storing metadata, currently unused).

Let's focus on just the `libraries` view, since it is a bit more complicated than `transport`:

In [10]:
thermo_df = read_sql("thermo_libraries_view", thermo_db).set_index("id")
thermo_df.head(2)

,name,short_description,long_description,label,adjacency_list,Tdata_unit,Cpdata_unit,H298,H298_unit,S298,...,c1,c2,c3,c4,c5,c6,c7,poly_Tmin,poly_Tmax,poly_T_unit
id,,,,,,,,,,,,,,,,,,,,,
0,GRI-Mech3.0-N,,,C(T),\nmultiplicity 3\n1 C u2 p1 c0\n,K,cal/(mol*K),171.271,kcal/mol,37.7801,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,GRI-Mech3.0-N,,,C2H,"\nmultiplicity 2\n1 H u0 p0 c0 {2,S}\n2 C u0 p...",K,cal/(mol*K),135.310,kcal/mol,50.9787,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [11]:
print(thermo_df.columns)

Index(['name', 'short_description', 'long_description', 'label',
       'adjacency_list', 'Tdata_unit', 'Cpdata_unit', 'H298', 'H298_unit',
       'S298', 'S298_unit', 'Tdata_1', 'Tdata_2', 'Tdata_3', 'Tdata_4',
       'Tdata_5', 'Tdata_6', 'Tdata_7', 'Cpdata_1', 'Cpdata_2', 'Cpdata_3',
       'Cpdata_4', 'Cpdata_5', 'Cpdata_6', 'Cpdata_7', 'nasa_Tmin',
       'nasa_Tmax', 'nasa_T_unit', 'E0', 'E0_unit', 'Cp0', 'Cp0_unit', 'CpInf',
       'CpInf_unit', 'c1', 'c2', 'c3', 'c4', 'c5', 'c6', 'c7', 'poly_Tmin',
       'poly_Tmax', 'poly_T_unit'],
      dtype='str')


Generally speaking, records in the thermo library have _either_ experimental data (e.g., H298) _or_ a NASA polynomial (e.g., NASA_Tmax).
We can select each of these in Pandas:

In [12]:
thermo_data_df = thermo_df[thermo_df['H298'].notna() & thermo_df['adjacency_list'].notna()].copy().dropna(axis='columns', how='all')
thermo_data_df.head(2)

,name,short_description,long_description,label,adjacency_list,Tdata_unit,Cpdata_unit,H298,H298_unit,S298,...,Tdata_5,Tdata_6,Tdata_7,Cpdata_1,Cpdata_2,Cpdata_3,Cpdata_4,Cpdata_5,Cpdata_6,Cpdata_7
id,,,,,,,,,,,,,,,,,,,,,
0,GRI-Mech3.0-N,,,C(T),\nmultiplicity 3\n1 C u2 p1 c0\n,K,cal/(mol*K),171.271,kcal/mol,37.7801,...,800.0,1000.0,1500.0,4.9798,4.9734,4.9715,4.9711,4.9692,4.9691,4.9742
1,GRI-Mech3.0-N,,,C2H,"\nmultiplicity 2\n1 H u0 p0 c0 {2,S}\n2 C u0 p...",K,cal/(mol*K),135.310,kcal/mol,50.9787,...,800.0,1000.0,1500.0,10.0485,10.5393,10.8828,11.1958,11.9369,12.6543,14.1032


There is an additional step for the NASA polynomials: each species can (and usually _does_) have multiple NSA polynomials that are valid at different temperature ranges.
One may wish to simply group these together, or perhaps select only the polynomials valid at a certain temperature.

See the [RMG docs](https://reactionmechanismgenerator.github.io/RMG-Py/reference/thermo/nasa.html) for more information about NASA estimations.

In [13]:
thermo_nasa_df = thermo_df[thermo_df['nasa_Tmin'].notna() & thermo_df['adjacency_list'].notna()].copy().dropna(axis='columns', how='all')
thermo_nasa_df.head(5)

,name,short_description,long_description,label,adjacency_list,nasa_Tmin,nasa_Tmax,nasa_T_unit,E0,E0_unit,c1,c2,c3,c4,c5,c6,c7,poly_Tmin,poly_Tmax,poly_T_unit
id,,,,,,,,,,,,,,,,,,,,
108,SABIC_aromatics,,,C4H3O2_6,"\nmultiplicity 2\n1 O u0 p2 c0 {5,S} {6,S}\n2 ...",200.0,3200.0,K,NaN,NaN,1.67401,0.027853,1.789220e-06,-2.006550e-08,8.733120e-12,-19258.0,18.07000,200.00,1029.77,K
108,SABIC_aromatics,,,C4H3O2_6,"\nmultiplicity 2\n1 O u0 p2 c0 {5,S} {6,S}\n2 ...",200.0,3200.0,K,NaN,NaN,6.24477,0.024570,-1.450880e-05,4.132870e-09,-4.547320e-13,-20966.6,-7.84094,1029.77,3200.00,K
109,SABIC_aromatics,,,C4H3O2_6,"\nmultiplicity 2\n1 O u0 p2 c0 {5,S} {6,S}\n2 ...",200.0,3200.0,K,NaN,NaN,1.67401,0.027853,1.789220e-06,-2.006550e-08,8.733120e-12,-19258.0,18.07000,200.00,1029.77,K
109,SABIC_aromatics,,,C4H3O2_6,"\nmultiplicity 2\n1 O u0 p2 c0 {5,S} {6,S}\n2 ...",200.0,3200.0,K,NaN,NaN,6.24477,0.024570,-1.450880e-05,4.132870e-09,-4.547320e-13,-20966.6,-7.84094,1029.77,3200.00,K
110,SABIC_aromatics,,,C4H5_1,"\nmultiplicity 2\n1 C u0 p0 c0 {2,S} {3,D} {5,...",200.0,3200.0,K,NaN,NaN,2.03117,0.025085,-9.411790e-07,-1.403020e-08,6.503880e-12,42100.2,15.64540,200.00,977.80,K


Use the `id` column for grouping together the multiple polynomials per species:

In [14]:
for id, sub_df in thermo_nasa_df.groupby('id'):
    print(sub_df)
    break

                name short_description long_description     label  \
id                                                                  
108  SABIC_aromatics                                     C4H3O2_6   
108  SABIC_aromatics                                     C4H3O2_6   

                                        adjacency_list  nasa_Tmin  nasa_Tmax  \
id                                                                             
108  \nmultiplicity 2\n1 O u0 p2 c0 {5,S} {6,S}\n2 ...      200.0     3200.0   
108  \nmultiplicity 2\n1 O u0 p2 c0 {5,S} {6,S}\n2 ...      200.0     3200.0   

    nasa_T_unit  E0 E0_unit       c1        c2        c3            c4  \
id                                                                       
108           K NaN     NaN  1.67401  0.027853  0.000002 -2.006550e-08   
108           K NaN     NaN  6.24477  0.024570 -0.000015  4.132870e-09   

               c5       c6        c7  poly_Tmin  poly_Tmax poly_T_unit  
id                              

Use filters to select at specific temperatures, e.g., between 1000 and 2000 K:

In [15]:
thermo_nasa_df[(thermo_nasa_df['poly_Tmin'] >= 1000) & (thermo_nasa_df['poly_Tmax'] <= 2000)].head(5)

,name,short_description,long_description,label,adjacency_list,nasa_Tmin,nasa_Tmax,nasa_T_unit,E0,E0_unit,c1,c2,c3,c4,c5,c6,c7,poly_Tmin,poly_Tmax,poly_T_unit
id,,,,,,,,,,,,,,,,,,,,
2082,2-BTP,MCMGOLest,\nMCMGOLest\n_low T polynomial Tmin changed fr...,CH2BR,"\nmultiplicity 2\n1 Br u0 p3 c0 {2,S}\n2 C u1...",298.0,1500.0,K,169.453,kJ/mol,-0.287987,0.010936,-0.000005,9.041900e-10,-6.286700e-14,20433.5,28.7153,1040.0,1500.0,K
2083,2-BTP,MCMGOLest,\nMCMGOLest\n_low T polynomial Tmin changed fr...,CH2BR,"\nmultiplicity 2\n1 Br u0 p3 c0 {2,S}\n2 C u1...",298.0,1500.0,K,169.453,kJ/mol,-0.287987,0.010936,-0.000005,9.041900e-10,-6.286700e-14,20433.5,28.7153,1040.0,1500.0,K
2148,SulfurGlarborgMarshall,HS-OH,\nALZ/GLA01 BOZ/R\n,HSOH,"\n1 S u0 p2 c0 {2,S} {3,S}\n2 O u0 p2 c0 {1,S}...",298.0,1500.0,K,NaN,NaN,2.567640,0.011380,-0.000006,-5.947000e-10,8.743830e-13,-15571.3,11.7664,1500.0,1500.0,K
2149,SulfurGlarborgMarshall,HS-OH,\nALZ/GLA01 BOZ/R\n,HSOH,"\n1 S u0 p2 c0 {2,S} {3,S}\n2 O u0 p2 c0 {1,S}...",298.0,1500.0,K,NaN,NaN,2.567640,0.011380,-0.000006,-5.947000e-10,8.743830e-13,-15571.3,11.7664,1500.0,1500.0,K
2150,SulfurGlarborgMarshall,HO-S*=O,\nDAG/GLA03 GOU/MAR99\n,HOSO,"\nmultiplicity 2\n1 O u0 p2 c0 {2,S} {4,S}\n2 ...",298.0,1500.0,K,NaN,NaN,1.618470,0.021164,-0.000027,1.627220e-08,-3.777900e-12,-30255.6,19.4773,1500.0,1500.0,K


## `solvation`

In [16]:
solvation_db = "demo_db/solvation.db"
list_all_views(solvation_db);

Views in database 'demo_db/solvation.db':
- label_pairs_view
- solute_groups_view
- solute_libraries_view
- solvent_libraries_view


This is also largely the same as the previous sub-databases, except that the libraries are split into two different views: solute and solvent.
There isn't any fundamental architectural reason for this, more just convenience for loading the two separately:

In [17]:
read_sql("solute_libraries_view", solvation_db).set_index("id")

,name,short_description,long_description,label,molecule,S,B,E,L,A,V
id,,,,,,,,,,,
0,solute,,"\nFrom Abarahm et al., J. Chem. Soc., Perkin T...",methane,C,0.000000,0.000000,0.000000,-0.323000,0.000000,0.2495
1,solute,,"\nFrom Abarahm et al., J. Chem. Soc., Perkin T...",ethane,CC,0.000000,0.000000,0.000000,0.492000,0.000000,0.3904
2,solute,,"\nFrom Abarahm et al., J. Chem. Soc., Perkin T...",propane,CCC,0.000000,0.000000,0.000000,1.050000,0.000000,0.5313
3,solute,,"\nFrom Abarahm et al., J. Chem. Soc., Perkin T...",n-butane,CCCC,0.000000,0.000000,0.000000,1.615000,0.000000,0.6722
4,solute,,"\nFrom Abarahm et al., J. Chem. Soc., Perkin T...",2-methylpropane,CC(C)C,0.000000,0.000000,0.000000,1.409000,0.000000,0.6722
...,...,...,...,...,...,...,...,...,...,...,...
445,solute,COSMO fit,\nGeometries from LithiumPrimaryThermo library...,[CH2]C#N,[CH2]C#N,0.722629,0.275263,0.360986,1.656567,0.086384,0.3827
446,solute,COSMO fit,\nGeometries from LithiumPrimaryThermo library...,[Li]N=[C]C,[Li]N=[C]C,1.451317,-0.278358,-1.267313,-2.600696,0.777030,0.5609
447,solute,COSMO fit,\nGeometries from LithiumPrimaryThermo library...,[Li]N=CC,[Li]N=CC,2.391183,1.099293,5.293384,10.955795,1.180529,0.5824


The solvent table has many more columns than the solutes:

In [18]:
df = read_sql("solvent_libraries_view", solvation_db).set_index("id")
df.columns

Index(['name', 'short_description', 'long_description', 'label', 'molecule',
       's_g', 'b_g', 'e_g', 'l_g', 'a_g', 'c_g', 's_h', 'b_h', 'e_h', 'l_h',
       'a_h', 'c_h', 'A', 'B', 'C', 'D', 'E', 'alpha', 'beta', 'eps', 'n',
       'name_in_coolprop', 'dGsolvCount', 'dGsolvMAE_val', 'dGsolvMAE_unit',
       'dHsolvCount', 'dHsolvMAE_val', 'dHsolvMAE_unit'],
      dtype='str')

In [19]:
df.head(5)

,name,short_description,long_description,label,molecule,s_g,b_g,e_g,l_g,a_g,...,beta,eps,n,name_in_coolprop,dGsolvCount,dGsolvMAE_val,dGsolvMAE_unit,dHsolvCount,dHsolvMAE_val,dHsolvMAE_unit
id,,,,,,,,,,,,,,,,,,,,,
0,solvent,,\nAbraham and Mintz parameters: fitted by Chun...,water,O,2.74983,4.84491,0.83346,-0.22544,3.92725,...,0.38,80.4,1.33300,water,5224.0,0.17,kcal/mol,58.0,1.04,kcal/mol
1,solvent,,"\nalpha = 0.328, #primary alcohols\nbeta = 0.4...",1-octanol,CCCCCCCCO,0.71369,1.42785,0.01254,0.85312,3.52275,...,0.45,10.3,1.42050,NaN,4189.0,0.21,kcal/mol,164.0,0.50,kcal/mol
2,solvent,,\nAbraham and Mintz parameters: fitted by Chun...,benzene,C1=CC=CC=C1,1.07490,0.17492,-0.32585,1.01356,0.56683,...,0.14,2.3,1.50110,benzene,110.0,0.19,kcal/mol,200.0,0.35,kcal/mol
3,solvent,,\nAbraham and Mintz parameters: fitted by Chun...,cyclohexane,C1CCCCC1,0.00000,-0.03443,-0.32662,1.03470,0.00000,...,0.00,2.0,1.42662,CycloHexane,122.0,0.22,kcal/mol,226.0,0.31,kcal/mol
4,solvent,,\nAbraham and Mintz parameters: fitted by Chun...,dibutylether,CCCCOCCCC,0.63588,-0.27257,-0.36266,0.98243,2.44885,...,0.45,3.1,1.39920,NaN,90.0,0.20,kcal/mol,78.0,0.27,kcal/mol


All of which can be filtered against, as done previously:

In [20]:
df[df['eps'].notna()][["label", "molecule", "eps"]].head(5)

,label,molecule,eps
id,,,
0,water,O,80.4
1,1-octanol,CCCCCCCCO,10.3
2,benzene,C1=CC=CC=C1,2.3
3,cyclohexane,C1CCCCC1,2.0
4,dibutylether,CCCCOCCCC,3.1


Notice that for these views, in this sub-database only, `rmgdatabase` does not use RMG's adjacency list format.
This is because `RMG-database`, for this data only, natively stores the structures as SMILES.

## `statmech`

Starting with the views:

In [21]:
statmech_db = "demo_db/statmech.db"
list_all_views(statmech_db);

Views in database 'demo_db/statmech.db':
- label_pairs_view
- statmech_groups_view
- statmech_libraries_view


This is another very standard sub-database!

In [22]:
df = read_sql("statmech_libraries_view", statmech_db).set_index("id")
df.head(3)

,name,short_description,long_description,label,adjacency_list,energy,energy_unit,spin_multiplicity,optical_isomers,mass,...,harmonic_freq_3,harmonic_freq_4,harmonic_freq_5,harmonic_freq_6,harmonic_freq_7,harmonic_freq_8,harmonic_freq_9,harmonic_freq_10,harmonic_freq_11,harmonic_freq_12
id,,,,,,,,,,,,,,,,,,,,,
0,halogens_G4,B3LYP/GTBas3,,HF,"\n1 F u0 p3 c0 {2,S}\n2 H u0 p0 c0 {1,S}\n",-282.3080,kJ/mol,None,None,20.0062,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,halogens_G4,B3LYP/GTBas3,,HBr,"\n1 Br u0 p3 c0 {2,S}\n2 H u0 p0 c0 {1,S}\n",-42.7435,kJ/mol,None,None,79.9262,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,halogens_G4,B3LYP/GTBas3,,HCl,"\n1 Cl u0 p3 c0 {2,S}\n2 H u0 p0 c0 {1,S}\n",-99.1327,kJ/mol,None,None,35.9767,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


One important division in these data is the linear vs non-linear rotors - similar to the `thermo` databases, each of these can be selected separately by looking at their corresponding fields:

In [23]:
statmech_linear_df = df[df['linear_symmetry'].notna()].copy().dropna(axis='columns', how='all')
statmech_linear_df.head(5)


,name,short_description,long_description,label,adjacency_list,energy,energy_unit,mass,mass_unit,linear_inertia,linear_inertia_unit,linear_symmetry,harmonic_freq_unit,harmonic_freq_1,harmonic_freq_2,harmonic_freq_3,harmonic_freq_4,harmonic_freq_5,harmonic_freq_6,harmonic_freq_7
id,,,,,,,,,,,,,,,,,,,,
0,halogens_G4,B3LYP/GTBas3,,HF,"\n1 F u0 p3 c0 {2,S}\n2 H u0 p0 c0 {1,S}\n",-282.30800,kJ/mol,20.0062,amu,0.809097,amu*angstrom^2,1.0,cm^-1,4113.430,NaN,NaN,NaN,NaN,NaN,NaN
1,halogens_G4,B3LYP/GTBas3,,HBr,"\n1 Br u0 p3 c0 {2,S}\n2 H u0 p0 c0 {1,S}\n",-42.74350,kJ/mol,79.9262,amu,2.014440,amu*angstrom^2,1.0,cm^-1,2635.590,NaN,NaN,NaN,NaN,NaN,NaN
2,halogens_G4,B3LYP/GTBas3,,HCl,"\n1 Cl u0 p3 c0 {2,S}\n2 H u0 p0 c0 {1,S}\n",-99.13270,kJ/mol,35.9767,amu,1.614060,amu*angstrom^2,1.0,cm^-1,2956.350,NaN,NaN,NaN,NaN,NaN,NaN
3,halogens_G4,B3LYP/GTBas3,,F2,"\n1 F u0 p3 c0 {2,S}\n2 F u0 p3 c0 {1,S}\n",-5.50154,kJ/mol,37.9968,amu,18.298700,amu*angstrom^2,2.0,cm^-1,1076.600,NaN,NaN,NaN,NaN,NaN,NaN
4,halogens_G4,B3LYP/GTBas3,,FCl,"\n1 Cl u0 p3 c0 {2,S}\n2 F u0 p3 c0 {1,S}\n",-65.04860,kJ/mol,53.9673,amu,33.221000,amu*angstrom^2,1.0,cm^-1,788.172,NaN,NaN,NaN,NaN,NaN,NaN


In [24]:
statmech_nonlinear_df = df[df['linear_symmetry'].isna()].copy().dropna(axis='columns', how='all')
statmech_nonlinear_df.head(5)


,name,short_description,long_description,label,adjacency_list,energy,energy_unit,mass,mass_unit,inertia_x,...,harmonic_freq_3,harmonic_freq_4,harmonic_freq_5,harmonic_freq_6,harmonic_freq_7,harmonic_freq_8,harmonic_freq_9,harmonic_freq_10,harmonic_freq_11,harmonic_freq_12
id,,,,,,,,,,,,,,,,,,,,,
12,halogens_G4,B3LYP/GTBas3,,OF,"\n1 F u0 p3 c0 {2,S}\n2 O u0 p2 c0 {1,S} {3,S}...",-95.2653,kJ/mol,36.0011,amu,0.860315,...,3730.420,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
13,halogens_G4,B3LYP/GTBas3,,OBr,"\n1 Br u0 p3 c0 {2,S}\n2 O u0 p2 c0 {1,S} {3,...",-71.8729,kJ/mol,95.9211,amu,0.831251,...,3777.610,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
14,halogens_G4,B3LYP/GTBas3,,OCl,"\n1 Cl u0 p3 c0 {2,S}\n2 O u0 p2 c0 {1,S} {3,...",-84.4995,kJ/mol,51.9716,amu,0.832669,...,3767.440,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
15,halogens_G4,B3LYP/GTBas3,,FOF,"\n1 F u0 p3 c0 {3,S}\n2 F u0 p3 c0 {3,S}\n3 O ...",16.0257,kJ/mol,53.9917,amu,8.341350,...,1034.580,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
16,halogens_G4,B3LYP/GTBas3,,FOBr,"\n1 Br u0 p3 c0 {3,S}\n2 F u0 p3 c0 {3,S}\n3 ...",58.9402,kJ/mol,113.9120,amu,10.582500,...,907.824,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## `kinetics`

Finally, the kinetics sub-database.
This one is far and away the most complicated and the largest, so let's walk through it starting with the views:

In [25]:
kinetics_db = "demo_db/kinetics.db"
list_all_views(kinetics_db);

Views in database 'demo_db/kinetics.db':
- all_family_rules_kinetics_view
- all_family_training_kinetics_view
- all_library_kinetics_view
- kinetics_families_view
- kinetics_family_forbidden_groups_view
- kinetics_family_groups_view
- kinetics_family_training_dictionary_view
- kinetics_family_training_reaction_species_view
- kinetics_library_dictionary_view
- kinetics_library_reaction_species_view
- label_pairs_view


These are what each of these is:

 - `all_family_rules_kinetics_view`: contains the learned estimation rules for all of the kinetics families in RMG
 - `all_family_training_kinetics_view`: training reactions using to derive the rules in `all_family_rules_kinetics_view`, note that `*_family_*` views are given a `family_name` based on the RMG reaction type
 - `all_library_kinetics_view`: known reactions from the literature/quantum mechanics simulations, which are used as a starting point to derive training reactions
 - `kinetics_families_view`: the actual reaction families in RMG, including their archetypal transformation
 - `kinetics_family_forbidden_groups_view`: some reaction families forbid certain species from being included in mechanisms; this shows all of these
 - `kinetics_family_groups_view`: used in tandem with `label_pairs_view` to reconstruct the property estimation tree
 - `kinetics_family_training_dictionary_view`: maps a given family's labels into actual species adjacency lists, stored separately because of the variable number of reactants and products
 - `kinetics_family_training_reaction_species_view`: tracks which labels correspond to reactants and products within a given reaction, for a given training reaction
 - `kinetics_library_dictionary_view` and `kinetics_library_reaction_species_view`: same as the equivalent `kinetics_family_*_view`, except for libraries rather than families.

The most useful of these for data access purposes are the `all_library_kinetics_view` and `all_family_training_kinetics_view`, whereas the others are more useful from the RMG side for loading only subsets of the larger database in very specific ways.

For this demo, let's show how to pull out all of a specific reaction type like Troe and Arrhenius.

One can pull out only the columns from the database which are relevant to the kinetics type either from SQL directly, using our `read_sql` function we defined earlier:

In [26]:
df = read_sql("all_library_kinetics_view", kinetics_db, columns=["library_name", "adjacency_reaction", "overall_kinetics_type", "degeneracy", "arr_A_val", "arr_A_unit", "arr_n", "arr_Ea_val", "arr_Ea_unit", "arr_T0_val", "arr_T0_unit"])
df[df["overall_kinetics_type"] == "Arrhenius"]

,library_name,adjacency_reaction,overall_kinetics_type,degeneracy,arr_A_val,arr_A_unit,arr_n,arr_Ea_val,arr_Ea_unit,arr_T0_val,arr_T0_unit
4,2003_Miller_Propargyl_Recomb_High_P,"1 C u0 p0 c0 {2,S} {3,S} {7,S} {8,S}\n2 C u0...",Arrhenius,1.0,2.309000e+10,s^-1,0.360,34.586,kcal/mol,1.0,K
5,2003_Miller_Propargyl_Recomb_High_P,"1 C u0 p0 c0 {2,S} {5,D} {7,S}\n2 C u0 p0 c0...",Arrhenius,1.0,5.000000e+11,s^-1,0.056,29.257,kcal/mol,1.0,K
6,2003_Miller_Propargyl_Recomb_High_P,"1 C u0 p0 c0 {2,S} {5,D} {7,S}\n2 C u0 p0 c0...",Arrhenius,1.0,1.162000e+12,s^-1,-0.046,38.474,kcal/mol,1.0,K
7,2003_Miller_Propargyl_Recomb_High_P,"1 C u0 p0 c0 {2,S} {6,S} {7,S} {8,S}\n2 C u0...",Arrhenius,1.0,8.067000e+10,s^-1,0.649,8.030,kcal/mol,1.0,K
8,2003_Miller_Propargyl_Recomb_High_P,"1 C u0 p0 c0 {2,S} {5,D} {7,S}\n2 C u0 p0 c0...",Arrhenius,1.0,2.084000e+09,s^-1,0.809,39.151,kcal/mol,1.0,K
...,...,...,...,...,...,...,...,...,...,...,...
20595,vinylCPD_H,"multiplicity 2\n1 C u0 p0 c0 {2,D} {3,S} {6,S...",Arrhenius,1.0,1.080000e+06,s^-1,1.990,25.200,kcal/mol,1.0,K
20596,vinylCPD_H,"multiplicity 2\n1 C u0 p0 c0 {2,S} {3,S} {5,S...",Arrhenius,1.0,5.720000e+08,s^-1,0.210,17.000,kcal/mol,1.0,K
20597,vinylCPD_H,"multiplicity 2\n1 C u0 p0 c0 {2,S} {3,S} {4,S...",Arrhenius,1.0,5.720000e+08,s^-1,0.210,17.000,kcal/mol,1.0,K
20598,vinylCPD_H,"1 C u0 p0 c0 {2,D} {3,D}\n2 C u0 p0 c0 {1,D}...",Arrhenius,1.0,2.100000e+09,cm^3/(mol*s),1.430,4.130,kcal/mol,1.0,K


Or instead simply load the entire dataframe and then perform column selection in Pandas:

In [27]:
df = read_sql("all_library_kinetics_view", kinetics_db)
df[df["overall_kinetics_type"] == "Troe"].dropna(axis='columns', how='all')

,library_name,reaction_id,label,adjacency_reaction,degeneracy,short_description,long_description,overall_kinetics_type,troe_alpha,troe_T3,troe_T1,troe_T2,troe_high_A,troe_high_n,troe_high_Ea,troe_low_A,troe_low_n,troe_low_Ea
64,BurkeH2O2inArHe,63,H + O2 <=> HO2,multiplicity 2\n1 H u1 p0 c0\n\n + \nmultiplic...,1.0,MAIN BATH GAS IS Ar or He,"\nHigh-pressure limit from Troe, Proc. Comb. I...",Troe,0.500,1.000000e-30,1.000000e+30,NaN,4.650840e+12,0.44,0.0,9.042000e+19,-1.500,492.2
71,BurkeH2O2inArHe,69,H2O2 <=> OH + OH,"1 O u0 p2 c0 {2,S} {3,S}\n2 O u0 p2 c0 {1,S} {...",1.0,"Troe, Combust. Flame, 158:594-601 (2011)",\nRate constant is for Ar\nEfficiencies for H2...,Troe,0.430,1.000000e-30,1.000000e+30,NaN,2.000000e+12,0.90,48749.0,2.490000e+24,-2.300,48749.0
97,BurkeH2O2inN2,93,H + O2 <=> HO2,multiplicity 2\n1 H u1 p0 c0\n\n + \nmultiplic...,1.0,MAIN BATH GAS IS N2,"\nHigh-pressure limit from Troe, Proc. Comb. I...",Troe,0.500,1.000000e-30,1.000000e+30,NaN,4.650840e+12,0.44,0.0,6.366000e+20,-1.720,524.8
104,BurkeH2O2inN2,99,H2O2 <=> OH + OH,"1 O u0 p2 c0 {2,S} {3,S}\n2 O u0 p2 c0 {1,S} {...",1.0,"Troe, Combust. Flame, 158:594-601 (2011)",\nRate constant is for Ar\nEfficiencies for H2...,Troe,0.430,1.000000e-30,1.000000e+30,NaN,2.000000e+12,0.90,48749.0,2.490000e+24,-2.300,48749.0
297,CF2BrCl,289,H + O2 <=> HO2,multiplicity 2\n1 H u1 p0 c0\n\n + \nmultiplic...,1.0,The chemkin file reaction is H + O2 <=> HO2,,Troe,0.670,1.000000e-30,1.000000e+30,1.000000e+30,4.650000e+12,0.44,0.0,1.737000e+19,-1.230,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
20457,primarySulfurLibrary,20053,SO2 + H <=> HOSO,"1 S u0 p1 c0 {2,D} {3,D}\n2 O u0 p2 c0 {1,D}\n...",2.0,[Pilling2006],"\nPart of the ""SOx"" mechanism\nT range: 300-17...",Troe,0.283,2.720000e+02,3.995000e+03,NaN,2.590000e+12,1.63,7339.0,1.140000e+22,-6.140,11075.0
20458,primarySulfurLibrary,20054,SO2 + H <=> HSO2,"1 S u0 p1 c0 {2,D} {3,D}\n2 O u0 p2 c0 {1,D}\n...",1.0,[Pilling2006],"\nPart of the ""SOx"" mechanism\nT range: 200-10...",Troe,0.390,1.670000e+02,2.191000e+03,NaN,4.610000e+12,1.59,2472.0,1.970000e+18,-5.190,4513.0
20471,primarySulfurLibrary,20067,HOSO <=> SO + OH,"multiplicity 2\n1 O u0 p2 c0 {2,S} {4,S}\n2 S ...",1.0,[Pilling2002b],"\nPart of the ""SOx"" subset\nRRKM\nAlso availab...",Troe,0.950,2.989000e+03,1.100000e+00,NaN,9.940000e+21,-2.54,75891.0,1.160000e+46,-9.020,52953.0
20509,primarySulfurLibrary,20105,S + C2H2 <=> HCCS + H,multiplicity 3\n1 S u2 p2 c0\n\n + \n1 C u0 p0...,2.0,[Marshall2015a],"\nPart of the ""C-S"" mechanism\nT range: 300-10...",Troe,0.600,1.000000e-30,1.000000e+30,NaN,1.260000e+13,0.00,2677.0,3.600000e+29,-3.550,3955.0


`all_family_training_kinetics_view` is the same as above, except it is sorted by RMG reaction family instead of library:

In [28]:
df = read_sql("all_family_training_kinetics_view", kinetics_db)
df[df["family_name"] == "R_Recombination"].dropna(axis='columns', how='all')

""


The `adjacency_reaction` columns contains the adjacency lists for each of the species, separated by the same separators as the `label`.